In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Penanda aras litar dinamik dengan pasangan Bell yang dipotong

*Anggaran penggunaan: 22 saat pada pemproses Heron r2 (NOTA: Ini adalah anggaran sahaja. Masa jalan sebenar anda mungkin berbeza.)*
## Latar belakang
Perkakasan kuantum biasanya terhad kepada interaksi tempatan, tapi banyak algoritma memerlukan pembelitan Qubit yang jauh atau bahkan [Qubit pada pemproses berasingan](#references). Litar dinamik — iaitu litar dengan pengukuran pertengahan litar dan suap balik ke depan — menyediakan cara untuk mengatasi batasan ini dengan menggunakan komunikasi klasik masa nyata untuk melaksanakan operasi kuantum bukan setempat secara berkesan. Dalam pendekatan ini, hasil pengukuran dari satu bahagian litar (atau satu QPU) boleh mencetuskan Gate secara bersyarat pada bahagian lain, membolehkan kita mengangkut pembelitan merentasi jarak jauh. Ini membentuk asas skim **operasi setempat dan komunikasi klasik (LOCC)**, di mana kita menggunakan keadaan sumber yang terikat (pasangan Bell) dan berkomunikasi hasil pengukuran secara klasik untuk menghubungkan Qubit yang jauh.

Salah satu kegunaan LOCC yang menjanjikan ialah untuk merealisasikan Gate CNOT jarak jauh maya melalui teleportasi, seperti yang ditunjukkan dalam [tutorial pembelitan jarak jauh](/tutorials/long-range-entanglement). Daripada CNOT jarak jauh terus (yang mungkin tidak dibenarkan oleh sambungan perkakasan), kita cipta pasangan Bell dan lakukan pelaksanaan Gate berasaskan teleportasi. Walau bagaimanapun, ketepatan operasi sedemikian bergantung pada ciri-ciri perkakasan. Penyahaskoheranan Qubit semasa kelewatan yang diperlukan (semasa menunggu hasil pengukuran) dan kependaman komunikasi klasik boleh merendahkan keadaan yang terikat. Selain itu, ralat pada pengukuran pertengahan litar lebih sukar untuk diperbetulkan berbanding ralat pada pengukuran akhir kerana ia merambat ke seluruh litar melalui Gate bersyarat.

Dalam [eksperimen rujukan](#references), para pengarang memperkenalkan penanda aras ketepatan pasangan Bell untuk mengenal pasti bahagian mana dalam peranti yang paling sesuai untuk pembelitan berasaskan LOCC. Ideanya ialah untuk menjalankan litar dinamik kecil pada setiap kumpulan empat Qubit yang bersambung dalam pemproses. Litar empat-Qubit ini pertama sekali mencipta pasangan Bell pada dua Qubit tengah, kemudian menggunakannya sebagai sumber untuk membelit dua Qubit tepi menggunakan LOCC. Secara konkrit, Qubit 1 dan 2 disediakan ke dalam pasangan Bell tidak dipotong secara setempat (menggunakan Hadamard dan CNOT), dan kemudian rutin teleportasi menggunakan pasangan Bell itu untuk membelit Qubit 0 dan 3. Qubit 1 dan 2 diukur semasa pelaksanaan litar, dan berdasarkan hasil tersebut, pembetulan Pauli (X pada Qubit 3 dan Z pada Qubit 0) dikenakan. Qubit 0 dan 3 kemudiannya ditinggalkan dalam keadaan Bell di penghujung litar.

Untuk mengukur kualiti pasangan terikat akhir ini, kita ukur penstabilnya: khususnya, pariti dalam asas $Z$ ($Z_0Z_3$) dan dalam asas $X$ ($X_0X_3$). Untuk pasangan Bell yang sempurna, kedua-dua jangkaan ini bersamaan +1. Dalam amalan, hingar perkakasan akan mengurangkan nilai-nilai ini. Oleh itu, kita ulang litar dua kali untuk setiap pasangan Qubit: satu litar mengukur Qubit 0 dan 3 dalam asas $Z$, dan satu lagi mengukurnya dalam asas $X$. Daripada hasilnya, kita mendapat anggaran $\langle Z_0Z_3\rangle$ dan $\langle X_0X_3\rangle$ untuk pasangan Qubit tersebut. Kita gunakan min kuasa dua ralat (MSE) penstabil ini berbanding nilai ideal (1) sebagai metrik mudah untuk ketepatan pembelitan. MSE yang lebih rendah bermakna dua Qubit mencapai keadaan Bell yang lebih hampir kepada ideal (ketepatan lebih tinggi), manakala MSE yang lebih tinggi menunjukkan lebih banyak ralat. Dengan mengimbas eksperimen ini merentasi peranti, kita boleh menanda aras keupayaan pengukuran-dan-suap-balik kumpulan Qubit yang berbeza dan mengenal pasti pasangan Qubit terbaik untuk operasi LOCC.

Tutorial ini menunjukkan eksperimen pada peranti IBM Quantum&reg; untuk menggambarkan cara litar dinamik boleh digunakan untuk menjana dan menilai pembelitan antara Qubit yang jauh. Kita akan memetakan semua rantai empat-Qubit linear pada peranti, menjalankan litar teleportasi pada setiap satunya, kemudian menggambarkan taburan nilai MSE. Prosedur hujung-ke-hujung ini menunjukkan cara memanfaatkan Qiskit Runtime dan ciri litar dinamik untuk memaklumkan pilihan yang sedar perkakasan bagi memotong litar atau mengedarkan algoritma kuantum merentasi sistem modular.
## Keperluan
Sebelum memulakan tutorial ini, pastikan anda telah memasang yang berikut:

* Qiskit SDK v2.0 atau lebih baharu, dengan sokongan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.40 atau lebih baharu (`pip install qiskit-ibm-runtime`)
## Persediaan

In [ ]:
from qiskit import QuantumCircuit

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler import generate_preset_pass_manager

import numpy as np
import matplotlib.pyplot as plt


def create_bell_stab(initial_layouts):
    """
    Create a circuit for a 1D chain of qubits (number of qubits must be a multiple of 4),
    where a middle Bell pair is consumed to create a Bell at the edge.
    Takes as input a list of lists, where each element of the list is a
    1D chain of physical qubits that is used as the initial_layout for the transpiled circuit.
    Returns a list of length-2 tuples, each tuple contains a circuit to measure the ZZ stabilizer and
    a circuit to measure the XX stabilizer of the edge Bell state.
    """
    bell_circuits = []
    for (
        initial_layout
    ) in initial_layouts:  # Iterate over chains of physical qubits
        assert (
            len(initial_layout) % 4 == 0
        ), f"The length of the chain must be a multiple of 4, len(inital_layout)={len(initial_layout)}"
        num_pairs = len(initial_layout) // 4

        bell_parallel = QuantumCircuit(4 * num_pairs, 4 * num_pairs)

        for pair_idx in range(num_pairs):
            (q0, q1, q2, q3) = (
                pair_idx * 4,
                pair_idx * 4 + 1,
                pair_idx * 4 + 2,
                pair_idx * 4 + 3,
            )
            (c0, c1) = pair_idx * 4, pair_idx * 4 + 3  # edge qubits
            (ca0, ca1) = pair_idx * 4 + 1, pair_idx * 4 + 2  # middle qubits

            bell_parallel.h(q0)
            bell_parallel.h(q1)
            bell_parallel.cx(q1, q2)
            bell_parallel.cx(q0, q1)
            bell_parallel.cx(q2, q3)
            bell_parallel.h(q2)

        # add barrier BEFORE measurements and add id in conditional
        bell_parallel.barrier()
        for pair_idx in range(num_pairs):
            (q0, q1, q2, q3) = (
                pair_idx * 4,
                pair_idx * 4 + 1,
                pair_idx * 4 + 2,
                pair_idx * 4 + 3,
            )
            (ca0, ca1) = pair_idx * 4 + 1, pair_idx * 4 + 2  # middle qubits

            bell_parallel.measure(q1, ca0)
            bell_parallel.measure(q2, ca1)
        # bell_parallel.barrier() #remove barrier after measurement

        for pair_idx in range(num_pairs):
            (q0, q1, q2, q3) = (
                pair_idx * 4,
                pair_idx * 4 + 1,
                pair_idx * 4 + 2,
                pair_idx * 4 + 3,
            )
            (ca0, ca1) = pair_idx * 4 + 1, pair_idx * 4 + 2  # middle qubits
            with bell_parallel.if_test((ca0, 1)):
                bell_parallel.x(q3)
            with bell_parallel.if_test((ca1, 1)):
                bell_parallel.z(q0)
                bell_parallel.id(q0)  # add id here for correct alignment

        bell_zz = bell_parallel.copy()
        bell_zz.barrier()
        bell_xx = bell_parallel.copy()
        bell_xx.barrier()
        for pair_idx in range(num_pairs):
            (q0, q1, q2, q3) = (
                pair_idx * 4,
                pair_idx * 4 + 1,
                pair_idx * 4 + 2,
                pair_idx * 4 + 3,
            )
            bell_xx.h(q0)
            bell_xx.h(q3)
        bell_xx.barrier()
        for pair_idx in range(num_pairs):
            (q0, q1, q2, q3) = (
                pair_idx * 4,
                pair_idx * 4 + 1,
                pair_idx * 4 + 2,
                pair_idx * 4 + 3,
            )
            (c0, c1) = pair_idx * 4, pair_idx * 4 + 3  # edge qubits

            bell_zz.measure(q0, c0)
            bell_zz.measure(q3, c1)

            bell_xx.measure(q0, c0)
            bell_xx.measure(q3, c1)

        bell_circuits.append(bell_zz)
        bell_circuits.append(bell_xx)

    return bell_circuits


def get_mse(result, initial_layouts):
    """
    given a result object and the initial layouts, returns a dict of layouts and their mse
    """
    layout_mse = {}
    for layout_idx, initial_layout in enumerate(initial_layouts):
        layout_mse[tuple(initial_layout)] = {}

        num_pairs = len(initial_layout) // 4

        counts_zz = result[2 * layout_idx].data.c.get_counts()
        total_shots = sum(counts_zz.values())

        # Get ZZ expectation value
        exp_zz_list = []
        for pair_idx in range(num_pairs):
            exp_zz = 0
            for bitstr, shots in counts_zz.items():
                bitstr = bitstr[::-1]  # reverse order to big endian
                b1, b0 = (
                    bitstr[pair_idx * 4],
                    bitstr[pair_idx * 4 + 3],
                )  # parse bitstring to get edge measurements for each 4-q chain
                z_val0 = 1 if b0 == "0" else -1
                z_val1 = 1 if b1 == "0" else -1
                exp_zz += z_val0 * z_val1 * shots
            exp_zz /= total_shots
            exp_zz_list.append(exp_zz)

        counts_xx = result[2 * layout_idx + 1].data.c.get_counts()
        total_shots = sum(counts_xx.values())

        # Get XX expectation value
        exp_xx_list = []
        for pair_idx in range(num_pairs):
            exp_xx = 0
            for bitstr, shots in counts_xx.items():
                bitstr = bitstr[::-1]  # reverse order to big endian
                b1, b0 = (
                    bitstr[pair_idx * 4],
                    bitstr[pair_idx * 4 + 3],
                )  # parse bitstring to get edge measurements for each 4-q chain
                x_val0 = 1 if b0 == "0" else -1
                x_val1 = 1 if b1 == "0" else -1
                exp_xx += x_val0 * x_val1 * shots
            exp_xx /= total_shots
            exp_xx_list.append(exp_xx)

        mse_list = [
            ((exp_zz - 1) ** 2 + (exp_xx - 1) ** 2) / 2
            for exp_zz, exp_xx in zip(exp_zz_list, exp_xx_list)
        ]

        print(f"layout {initial_layout}")
        for idx in range(num_pairs):
            layout_mse[tuple(initial_layout)][
                tuple(initial_layout[4 * idx : 4 * idx + 4])
            ] = mse_list[idx]
            print(
                f"qubits: {initial_layout[4*idx:4*idx+4]}, mse:, {round(mse_list[idx],4)}"
            )
            # print(f'exp_zz: {round(exp_zz_list[idx],4)}, exp_xx: {round(exp_xx_list[idx],4)}')
        print(" ")
    return layout_mse


def plot_mse_ecdfs(layouts_mse, combine_layouts=False):
    """
    Plot CDF of MSE data for multiple layouts. Optionally combine all data in a single CDF
    """

    if not combine_layouts:
        for initial_layout, layouts in layouts_mse.items():
            sorted_layouts = dict(
                sorted(layouts.items(), key=lambda item: item[1])
            )  # sort layouts by mse

            # get layouts and mses
            layout_list = list(sorted_layouts.keys())
            mse_list = np.asarray(list(sorted_layouts.values()))

            # convert to numpy
            x = np.array(mse_list)
            y = np.arange(1, len(x) + 1) / len(x)

            # Prepend (x[0], 0) to start CDF at zero
            x = np.insert(x, 0, x[0])
            y = np.insert(y, 0, 0)

            # Create the plot
            plt.plot(
                x,
                y,
                marker="x",
                linestyle="-",
                label=f"qubits: {initial_layout}",
            )

            # add qubits labels for the edge pairs
            for xi, yi, q in zip(x[1:], y[1:], layout_list):
                plt.annotate(
                    [q[0], q[3]],
                    (xi, yi),
                    textcoords="offset points",
                    xytext=(5, -10),
                    ha="left",
                    fontsize=8,
                )

    elif combine_layouts:
        all_layouts = {}
        all_initial_layout = []
        for (
            initial_layout,
            layouts,
        ) in layouts_mse.items():  # puts together all layout information
            all_layouts.update(layouts)
            all_initial_layout += initial_layout

        sorted_layouts = dict(
            sorted(all_layouts.items(), key=lambda item: item[1])
        )  # sort layouts by mse

        # get layouts and mses
        layout_list = list(sorted_layouts.keys())
        mse_list = np.asarray(list(sorted_layouts.values()))

        # convert to numpy
        x = np.array(mse_list)
        y = np.arange(1, len(x) + 1) / len(x)

        # Prepend (x[0], 0) to start CDF at zero
        x = np.insert(x, 0, x[0])
        y = np.insert(y, 0, 0)

        # Create the plot
        plt.plot(
            x,
            y,
            marker="x",
            linestyle="-",
            label=f"qubits: {sorted(list(set(all_initial_layout)))}",
        )

        # add qubit labels for the edge pairs
        for xi, yi, q in zip(x[1:], y[1:], layout_list):
            plt.annotate(
                [q[0], q[3]],
                (xi, yi),
                textcoords="offset points",
                xytext=(5, -10),
                ha="left",
                fontsize=8,
            )

    plt.xscale("log")
    plt.xlabel("Mean squared error of ⟨ZZ⟩ and ⟨XX⟩")
    plt.ylabel("Cumulative distribution function")
    plt.title("CDF for different initial layouts")
    plt.grid(alpha=0.3)
    plt.show()

## Langkah 1: Petakan input klasik kepada masalah kuantum
Langkah pertama adalah mencipta satu set Circuit kuantum untuk menilai semua calon pautan Bell-pair yang disesuaikan dengan topologi peranti. Kita mencari secara aturcara peta gandingan peranti untuk semua rantai empat Qubit yang bersambung secara linear. Setiap rantai sedemikian (dilabel dengan indeks Qubit $[q0-q1-q2-q3]$) berfungsi sebagai kes ujian untuk Circuit pertukaran keterikatan. Dengan mengenal pasti semua laluan panjang-4 yang mungkin, kita memastikan liputan maksimum untuk pengelompokan Qubit yang boleh merealisasikan protokol ini.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(operational=True)

Kita menjana rantai-rantai ini dengan menggunakan fungsi pembantu yang melakukan carian tamak pada graf peranti. Ia mengembalikan "jalur" empat rantai empat-Qubit yang dibundel kepada kumpulan 16-Qubit (Circuit dinamik pada masa ini mengehadkan saiz daftar pengukuran kepada `16` Qubit). Pembundelan membolehkan kita menjalankan pelbagai eksperimen empat-Qubit secara selari pada bahagian cip yang berbeza, dan menggunakan keseluruhan peranti dengan cekap. Setiap jalur 16-Qubit mengandungi empat rantai yang tidak bertindih, bermakna tiada Qubit yang digunakan semula dalam kumpulan tersebut. Sebagai contoh, satu jalur mungkin terdiri daripada rantai $[0-1-2-3]$, $[4-5-6-7]$, $[8-9-10-11]$, dan $[12-13-14-15]$ semuanya dibungkus bersama. Sebarang Qubit yang tidak disertakan dalam jalur dikembalikan dalam pembolehubah `leftover`.

In [79]:
from itertools import chain
from collections import defaultdict


def stripes16_from_backend(backend):
    """
    Creates stripes of 16 qubits, four non-overlapping  four-qubit chains, that cover as much of
    the coupling map as possible. Returns any unused qubits as leftovers.
    """
    # get the undirected adjacency list
    edges = backend.coupling_map.get_edges()
    graph = defaultdict(set)
    for u, v in edges:
        graph[u].add(v)
        graph[v].add(u)

    qubits = sorted(graph)  # all qubit indices that appear

    # greedy search for 4-long linear chains (blocks) ────────────
    used = set()  # qubits already placed in a block
    blocks = []  # each block is a four-qubit list

    for q in qubits:  # deterministic order for reproducibility
        if q in used:
            continue  # already consumed by earlier block

        # depth-first "straight" walk of length 3 without revisiting nodes
        def extend(path):
            if len(path) == 4:
                return path
            tip = path[-1]
            for nbr in sorted(graph[tip]):  # deterministic
                if nbr not in path and nbr not in used:
                    maybe = extend(path + [nbr])
                    if maybe:
                        return maybe
            return None

        block = extend([q])
        if block:  # found a 4-node path
            blocks.append(block)
            used.update(block)

    # bundle four four-qubit blocks into one 16-qubit stripe (max number of measurement compatible with if-else)
    stripes = [
        list(chain.from_iterable(blocks[i : i + 4]))
        for i in range(0, len(blocks) // 4 * 4, 4)  # full groups of four
    ]

    leftovers = set(qubits) - set(chain.from_iterable(stripes))
    return stripes, leftovers

In [80]:
initial_layouts, leftover = stripes16_from_backend(backend)

Seterusnya, kita membina Circuit untuk setiap jalur 16-Qubit. Rutin ini melakukan perkara berikut untuk setiap rantai:

* Sediakan pasangan Bell tengah: Gunakan Hadamard pada Qubit 1 dan CNOT dari Qubit 1 ke Qubit 2. Ini melilit Qubit 1 dan 2 (mewujudkan keadaan Bell $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$).
* Lilit Qubit tepi: Gunakan CNOT dari Qubit 0 ke Qubit 1, dan CNOT dari Qubit 2 ke Qubit 3. Ini menghubungkan pasangan yang berasingan sebelum ini supaya Qubit 0 dan 3 akan menjadi terikat selepas langkah seterusnya. Hadamard pada Qubit 2 juga digunakan (ini, digabungkan dengan CNOT sebelumnya, membentuk sebahagian daripada pengukuran Bell pada Qubit 1 dan 2). Pada ketika ini, Qubit 0 dan 3 belum terikat, tetapi Qubit 1 dan 2 terikat dengan mereka dalam keadaan empat-Qubit yang lebih besar.
* Pengukuran litar tengah dan suap balas: Qubit 1 dan 2 (Qubit tengah) diukur dalam asas pengkomputeran, menghasilkan dua bit klasik. Berdasarkan keputusan pengukuran tersebut, kita menggunakan operasi bersyarat: jika pengukuran Qubit 1 (panggil bit ini $m_{12}$) adalah 1, kita gunakan Gate $X$ pada Qubit 3; jika pengukuran Qubit 2 ($m_{21}$) adalah 1, kita gunakan Gate $Z$ pada Qubit 0. Gate bersyarat ini (yang direalisasikan menggunakan konstruk `if_test`/`if_else` Qiskit) melaksanakan pembetulan teleportasi standard. Ia "membatalkan" balik Pauli rawak yang berlaku akibat memproyeksikan Qubit 1 dan 2, memastikan Qubit 0 dan 3 berakhir dalam keadaan Bell yang diketahui, tanpa mengira keputusan pengukuran. Selepas langkah ini, Qubit 0 dan 3 seharusnya secara idealnya terikat dalam keadaan Bell $|\Phi^+\rangle$.
* Ukur penstabil pasangan Bell: Kita kemudian membahagi kepada dua versi Circuit. Dalam versi pertama, kita mengukur penstabil $ZZ$ pada Qubit 0 dan 3. Dalam versi kedua, kita mengukur penstabil $XX$ pada Qubit-Qubit ini.

Untuk setiap susun atur awal empat-Qubit, fungsi di atas mengembalikan dua Circuit (satu untuk pengukuran penstabil $ZZ$, satu lagi untuk $XX$). Di penghujung langkah ini, kita mempunyai senarai Circuit yang merangkumi setiap rantai empat-Qubit pada peranti. Circuit-circuit ini termasuk pengukuran litar tengah dan operasi bersyarat (if/else), yang merupakan arahan utama Circuit dinamik.

In [63]:
circuits = create_bell_stab(initial_layouts)
circuits[-1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/edc-cut-bell-pair-benchmarking/extracted-outputs/bd04755f-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/edc-cut-bell-pair-benchmarking/extracted-outputs/bd04755f-0.avif)

## Langkah 2: Optimumkan masalah untuk pelaksanaan perkakasan kuantum
Sebelum melaksanakan Circuit kita pada perkakasan sebenar, kita perlu mentranspil circuit-circuit tersebut agar sepadan dengan kekangan fizikal peranti. Transpilasi akan memetakan Circuit abstrak kepada Qubit fizikal dan set Gate peranti yang dipilih. Memandangkan kita telah memilih Qubit fizikal tertentu untuk setiap rantai (dengan menyediakan `initial_layout` kepada penjana Circuit), kita menggunakan `optimization_level=0` Transpiler dengan susun atur tetap tersebut. Ini memberitahu Qiskit untuk tidak menetapkan semula Qubit atau melakukan sebarang pengoptimuman berat yang boleh mengubah struktur Circuit. Kita mahu mengekalkan urutan operasi (terutamanya Gate bersyarat) tepat seperti yang dinyatakan.

In [ ]:
isa_circuits = []
for ind, init_layout in enumerate(initial_layouts):
    pm = generate_preset_pass_manager(
        optimization_level=0, backend=backend, initial_layout=init_layout
    )
    isa_circ = pm.run(circuits[ind * 2 : ind * 2 + 2])
    isa_circuits.extend(isa_circ)

In [65]:
isa_circuits[1].draw("mpl", fold=-1, idle_wires=False)

<Image src="../docs/images/tutorials/edc-cut-bell-pair-benchmarking/extracted-outputs/3ad620f7-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/edc-cut-bell-pair-benchmarking/extracted-outputs/3ad620f7-0.avif)

## Langkah 3: Laksanakan menggunakan primitif Qiskit
Kita kini boleh menjalankan eksperimen pada peranti kuantum. Kita menggunakan Qiskit Runtime dan primitif Sampler untuk melaksanakan kelompok Circuit dengan cekap.

In [ ]:
sampler = Sampler(mode=backend)
sampler.options.environment.job_tags = ["cut-bell-pair-test"]
job = sampler.run(isa_circuits)

## Langkah 4: Proses selepas dan kembalikan keputusan dalam format klasik yang dikehendaki
Langkah terakhir adalah mengira metrik ralat min kuasa dua (MSE) untuk setiap kumpulan Qubit yang diuji dan merumuskan keputusan. Untuk setiap rantai, kita kini mempunyai $\langle Z_0Z_3\rangle$ dan $\langle X_0X_3\rangle$ yang diukur. Jika Qubit 0 dan 3 terikat sempurna dalam keadaan Bell $|\Phi^+\rangle$, kita menjangkakan kedua-duanya bernilai +1. Kita mengukur penyelewengan menggunakan MSE:

$$\text{MSE} = \frac{( \langle Z_0Z_3\rangle - 1)^2 + (\langle X_0X_3\rangle - 1)^2}{2}.$$

Nilai ini adalah 0 untuk pasangan Bell yang sempurna, dan meningkat apabila keadaan terikat menjadi lebih berderau (dengan keputusan rawak memberikan jangkaan sekitar 0, MSE akan menghampiri 1). Kod ini mengira MSE untuk setiap kumpulan empat-Qubit.

Keputusan mendedahkan julat kualiti keterikatan yang luas merentas peranti. Ini mengesahkan penemuan makalah bahawa terdapat variasi lebih daripada satu tertib magnitud dalam kesetiaan keadaan Bell bergantung pada Qubit fizikal yang digunakan. Dari segi praktikal, ini bermakna kawasan atau pautan tertentu dalam cip jauh lebih baik dalam melakukan operasi pengukuran litar tengah dan suap balas berbanding yang lain. Faktor seperti ralat bacaan Qubit, jangka hayat Qubit, dan crosstalk kemungkinan menyumbang kepada perbezaan ini. Sebagai contoh, jika satu rantai termasuk Qubit bacaan yang sangat berderau, pengukuran litar tengah boleh menjadi tidak boleh dipercayai, menyebabkan kesetiaan yang lemah untuk pasangan terikat tersebut (MSE tinggi).

In [71]:
layouts_mse = get_mse(job.result(), initial_layouts)

layout [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
qubits: [0, 1, 2, 3], mse:, 0.0312
qubits: [4, 5, 6, 7], mse:, 0.0491
qubits: [8, 9, 10, 11], mse:, 0.0711
qubits: [12, 13, 14, 15], mse:, 0.0436
 
layout [16, 23, 22, 21, 17, 27, 26, 25, 18, 31, 30, 29, 19, 35, 34, 33]
qubits: [16, 23, 22, 21], mse:, 0.0197
qubits: [17, 27, 26, 25], mse:, 0.113
qubits: [18, 31, 30, 29], mse:, 0.0287
qubits: [19, 35, 34, 33], mse:, 0.0433
 
layout [36, 41, 42, 43, 37, 45, 46, 47, 38, 49, 50, 51, 39, 53, 54, 55]
qubits: [36, 41, 42, 43], mse:, 0.1645
qubits: [37, 45, 46, 47], mse:, 0.0409
qubits: [38, 49, 50, 51], mse:, 0.0519
qubits: [39, 53, 54, 55], mse:, 0.0829
 
layout [56, 63, 62, 61, 57, 67, 66, 65, 58, 71, 70, 69, 59, 75, 74, 73]
qubits: [56, 63, 62, 61], mse:, 0.8663
qubits: [57, 67, 66, 65], mse:, 0.0375
qubits: [58, 71, 70, 69], mse:, 0.0664
qubits: [59, 75, 74, 73], mse:, 0.0291
 
layout [76, 81, 82, 83, 77, 85, 86, 87, 78, 89, 90, 91, 79, 93, 94, 95]
qubits: [76, 81, 82, 83], mse

Finally, we visualize the overall performance by plotting the cumulative distribution function (CDF) of the MSE values for all chains. The CDF plot shows the MSE threshold on the x-axis, and the fraction of qubit pairs that have at most that MSE on the y-axis. This curve starts at zero and approaches one as the threshold grows to encompass all data points. A steep rise near a low MSE would indicate that many pairs are high-fidelity; a slow rise means that many pairs have larger errors. We annotate the CDF with the identities of the best pairs. In the plot, each point in the CDF corresponds to one four-qubit chain's MSE, and we label the point with the pair of qubit indices $[q0, q3]$ that were entangled in that experiment. This makes it easy to spot which physical qubit pairs are the top performers (the far-left points on the CDF).

In [68]:
plot_mse_ecdfs(layouts_mse, combine_layouts=True)

<Image src="../docs/images/tutorials/edc-cut-bell-pair-benchmarking/extracted-outputs/678ddac9-0.avif" alt="Output of the previous code cell" />

Akhir sekali, kita menggambarkan prestasi keseluruhan dengan memplot fungsi taburan kumulatif (CDF) nilai MSE untuk semua rantai. Plot CDF menunjukkan ambang MSE pada paksi-x, dan pecahan pasangan Qubit yang mempunyai MSE paling banyak pada ambang tersebut di paksi-y. Lengkung ini bermula dari sifar dan menghampiri satu apabila ambang berkembang untuk merangkumi semua titik data. Kenaikan curam berhampiran MSE rendah menunjukkan bahawa banyak pasangan mempunyai kesetiaan tinggi; kenaikan perlahan bermakna banyak pasangan mempunyai ralat yang lebih besar. Kita memberi anotasi pada CDF dengan identiti pasangan terbaik. Dalam plot, setiap titik dalam CDF sepadan dengan MSE satu rantai empat-Qubit, dan kita melabelkan titik tersebut dengan pasangan indeks Qubit $[q0, q3]$ yang terikat dalam eksperimen tersebut. Ini memudahkan pengesanan pasangan Qubit fizikal mana yang merupakan pemain terbaik (titik paling kiri pada CDF).